# Logistics LP: Distillery Planning

CHBE444: Design I, Prof. Ganesh Sriram (gsriram@umd.edu), University of Maryland.

*The following LP was introduced in a past lecture.*

You are tasked with preparing a production plan for a distillery that produces two drinks—an ale (A) that fetches a profit of 5 dollars/gal, and a bourbon whiskey (B) that fetches a profit of 10 dollars/gal.

Four pieces of equipment are required for the production of both drinks—fermenter 1, fermenter 2,
fermenter 3 and still 4—each of which is available for 150 hours (h) per week. The production of 100 gal A requires 5 h in fermenter 1, 10 h in fermenter 2 and 5 h in fermenter 3. The production of 100 gal B requires 10 h in fermenter 1, 5 h in fermenter 2, 15 h in fermenter 3 and 20 h in still 4.

Our first task was to formulate an LP with an objective of maximizing the weekly profit of the distillery, which we did in a previous lecture. The LP was:

$$ 
\begin{array}{lrcl}
\max      & z = 5 x_1 + 10 x_2  & & \\
s.t.      & 0.05 x_1 + 0.10 x_2 & \le & 150 \\
          & 0.10 x_1 + 0.05 x_2 & \le & 150 \\
          & 0.05 x_1 + 0.15 x_2 & \le & 150 \\
          &            0.20 x_2 & \le & 150 \\
          & x_1, x_2            & \ge & 0 \\ 
\end{array}
$$

Our goal is to solve the LP to determine the optimal weekly profit and the corresponding production plan, i.e., how many gallons of each product should be produced per week.

Our main task is to solve the LP by using GAMSPy, which we will do below. To prepare for this goal, we will plot the feasible space of this LP and determine its optimum by method we already know, so we can test if GAMSPy gets the right answer.

Notes
+ This example was based on the standard GAMS example of a transportation problem: https://www.gams.com/54/docs/UG_Tutorial.html.
+ GAMSPy code for this and other example problems is available here: https://gamspy.readthedocs.io/en/latest/user/examples.html.
+ To learn more tricks for Jupyter markdown cells, see here: https://www.ibm.com/docs/en/db2-event-store/2.0.0?topic=notebooks-markdown-jupyter-cheatsheet.

## Desmos Plot of Feasible Space

In [1]:
import chbe444umd as des

s = 0.1  # shading opacity

expressions = [{'latex': 'x_1 = x', 'hidden': 'true'},
               {'latex': 'x_2 = y', 'hidden': 'true'},
               {'latex': '0.05 x_1 + 0.10 x_2 <= 150', 'id': 'f1',
                'fillOpacity': s, 'color': '#ffbe98'},
               {'latex': '0.10 x_1 + 0.05 x_2 <= 150', 'id': 'f2',
                'fillOpacity': s, 'color': '#ffbe98'},
               {'latex': '0.05 x_1 + 0.15 x_2 <= 150', 'id': 'f3',
                'fillOpacity': s, 'color': '#ffbe98'},
               {'latex': '0.20 x_2 <= 150', 'id': 's4',
                'fillOpacity': s, 'color': '#ffbe98'},
               {'latex': 'x_1 >= 0', 'id': 'x1',
                'fillOpacity': s, 'color': '#ffbe98'},
               {'latex': 'x_2 >= 0', 'id': 'x2',
                'fillOpacity': s, 'color': '#ffbe98'},
               {'latex': '5 x_1 + 10 x_2 = z', 'id': 'obj',
                'fillOpacity': s, 'color': '#000000'}]

polygon = ((0, 0), (1500, 0), (1200, 600), (750, 750), (0, 750))
feasible = [{'latex': '\\\\operatorname{polygon}' + str(polygon),
             'id': 'feasible', 'fillOpacity': 0, 'color': '#006167'}]
expressions.extend(feasible)

sliders = [{'variable': 'z',
            'min': '0',
            'max': '15000',
            'step': '1000'}]

points = [{'latex': '(1200,600)',
           'color': '#000000',
           'label': 'optimum',
           'showLabel': True}]

graph_html = des.desmos_graph(expressions=expressions,
                              sliders=sliders,
                              points=points,
                              axlim=[-100, 2000, -100, 2000],
                              size=[1000, 600])

<IPython.core.display.Javascript object>

## GAMSPy Solution

In [2]:
import numpy as np
import gamspy as gams
import pandas as pd

# For detailed instructions from GAMS, see:
# https://gamspy.readthedocs.io/en/latest/user/notebooks/trnsport.html


def print_records(vars):
    for k in np.arange(0, len(vars)):
        print(vars[k].name, ':')
        display(vars[k].records)
        print()


m = gams.Container()

i = gams.Set(container=m,
             name='i',
             description='reactor')

j = gams.Set(container=m,
             name='j',
             description='drink')

i.setRecords(['F1', 'F2', 'F3', 'S4'])
j.setRecords(['A', 'B'])

print_records([i, j])

capacity = gams.Parameter(
    container=m,
    name='capacity',
    domain=i,
    description='weekly production capacity of reactor i (h/wk)'
)

capacity.setRecords([('F1', 150),
                     ('F2', 150),
                     ('F3', 150),
                     ('S4', 150)])

profit = gams.Parameter(
    container=m,
    name='profit',
    domain=j,
    description='profit of drink j ($/gal)'
)

profit.setRecords([('A', 5),
                   ('B', 10)])

print_records([capacity, profit])

hours_matrix = pd.DataFrame(
    [
        ['F1', 'A', 0.05],
        ['F1', 'B', 0.10],
        ['F2', 'A', 0.10],
        ['F2', 'B', 0.05],
        ['F3', 'A', 0.05],
        ['F3', 'B', 0.15],
        ['S4', 'A', 0.00],
        ['S4', 'B', 0.20],
     ],
    columns=['reactor', 'drink', 'hours']
).set_index(['reactor', 'drink'])

hours = gams.Parameter(
    container=m,
    name='hours',
    domain=[i, j],
    description='number of hours spent in each reactor by each drink (h/gal)',
    records=hours_matrix.reset_index()
)

print_records([hours])

x = gams.Variable(
    container=m,
    name='x',
    domain=j,
    type='Positive',
    description='weekly production of drink j (gal/wk)'
)

capacity_constraint = gams.Equation(
    container=m,
    name='cap_constraint',
    domain=i,
    description='capacity of reactor i (h/wk)'
)

capacity_constraint[i] = gams.Sum(j, hours[i, j] * x[j]) <= capacity[i]

objective = gams.Sum(j, profit[j] * x[j])

problem = gams.Model(
    container=m,
    name='distillery',
    equations=[capacity_constraint],
    problem='LP',
    sense=gams.Sense.MAX,
    objective=objective
)

problem.solve(
    output=None,
    options=gams.Options(equation_listing_limit=10,
                         variable_listing_limit=10)
)

print(problem.getEquationListing())
print()
print('Optimal values of x:')
print_records([x])
print(f"Optimal value of z: {problem.objective_value}")

i :


,uni,element_text
0,F1,
1,F2,
2,F3,
3,S4,



j :


,uni,element_text
0,A,
1,B,



capacity :


...


hours :


,reactor,drink,value
0,F1,A,0.05
1,F1,B,0.10
2,F2,A,0.10
3,F2,B,0.05
4,F3,A,0.05
5,F3,B,0.15
6,S4,A,0.00
7,S4,B,0.20



cap_constraint(F1)..  0.05*x(A) + 0.1*x(B) =L= 150 ; (LHS = 0)
cap_constraint(F2)..  0.1*x(A) + 0.05*x(B) =L= 150 ; (LHS = 0)
cap_constraint(F3)..  0.05*x(A) + 0.15*x(B) =L= 150 ; (LHS = 0)
cap_constraint(S4)..  0.2*x(B) =L= 150 ; (LHS = 0)
distillery_objective..  5*x(A) + 10*x(B) - distillery_objective_variable =E= 0 ; (LHS = 0)

Optimal values of x:
x :


,j,level,marginal,lower,upper,scale
0,A,1200.0,0.0,0.0,inf,1.0
1,B,600.0,0.0,0.0,inf,1.0



Optimal value of z: 12000.0
